# Quickstart

The minimum viable fluxopt model: a gas boiler covers a 4-hour workshop heat demand.

```
Gas Grid ──[gas]──► Boiler (η=0.9) ──[heat]──► Workshop
0.08 €/kWh                                     50 kW peak
```

You'll meet the five pieces every fluxopt model needs: **Carrier**, **Flow**, **Port**, **Converter**, **Effect** — and the entry point: **optimize()**.

In [ ]:
from datetime import datetime

from fluxopt import Carrier, Converter, Effect, Flow, Port, optimize

## 1. Declare the system

A fluxopt model is built from these pieces:

- A **`Carrier`** is an energy type — `gas`, `heat`, `electricity`, …
- A **`Flow`** is energy moving on a carrier (in MW). Flows live on **`Port`**s (system boundaries: imports/exports) and **`Converter`**s (couple input flows to output flows).
- A **`Converter`** turns input flows into output flows via a conversion equation. `Converter.boiler(...)` is a shortcut for the standard relation `fuel · η = heat`.
- An **`Effect`** is anything you want to track or minimize: cost, CO₂, fuel use. Each `Flow` declares which effects it contributes to via `effects_per_flow_hour`.

In [ ]:
carriers = [Carrier('gas'), Carrier('heat')]
effects = [Effect('cost', unit='EUR')]

ports = [
    # Gas comes in from the grid, costs 0.08 EUR per kWh delivered.
    Port(
        'gas_grid',
        imports=[
            Flow('gas', size=1000, effects_per_flow_hour={'cost': 0.08}),
        ],
    ),
    # The workshop's heat demand: a fixed profile, scaled to 50 kW peak.
    Port(
        'workshop',
        exports=[
            Flow('heat', size=50, fixed_relative_profile=[0.6, 1.0, 0.9, 0.5]),
        ],
    ),
]

converters = [
    Converter.boiler(
        'boiler',
        thermal_efficiency=0.9,
        fuel_flow=Flow('gas', size=200),
        thermal_flow=Flow('heat', size=100),
    ),
]

## 2. Solve

`optimize()` builds the linear program, hands it to HiGHS, and returns a `Result` object. We minimize `'cost'` — the sum of `cost` contributions from every flow.

In [ ]:
result = optimize(
    timesteps=[datetime(2024, 1, 15, h) for h in range(8, 12)],
    carriers=carriers,
    effects=effects,
    ports=ports,
    converters=converters,
    objective='cost',
)

## 3. Read the result

The `Result` exposes solver outputs as `xarray` arrays. The most-used:

- `result.objective` — the minimized total (here, total cost in EUR)
- `result.effect_totals` — totals per tracked effect (1-D, `effect`)
- `result.flow_rates` — every flow's rate per timestep (2-D, `flow × time`, in MW)

In [ ]:
print(f'Total cost: {result.objective:.2f} EUR')
result.effect_totals.drop_sel(effect='penalty').to_dataframe(name='total')

All flow rates per timestep — notice how the boiler's gas input equals the heat output divided by 0.9, the conversion equation acting per row.


In [ ]:
result.flow_rates.to_pandas().T.round(2)

## Recap

You declared five kinds of object — `Carrier`, `Flow`, `Port`, `Converter`, `Effect` — passed them to `optimize()`, and read the solution as xarray.

Note flow ids: by default a flow's id is qualified as `component(carrier)` — `boiler(gas)`, `workshop(heat)`. That keeps flow names unique even when several flows share a carrier.

**Next**: [Storage](02-storage.ipynb) — adding a thermal storage to the same workshop.